# Text-to-Video Generation Lab (ZeroScope)

Inference-only practical: no dataset, training, fine-tuning, or API key. Select a **GPU runtime** before running. The checkpoint is public and ungated under **CC-BY-NC-4.0 (non-commercial use only)**. Model card: https://huggingface.co/cerspense/zeroscope_v2_576w


In [ ]:
import platform, sys, torch
print('Python:', sys.version)
print('Platform:', platform.platform())
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('GPU memory (GB):', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))
else:
    raise RuntimeError('CUDA GPU not available. Select Runtime > Change runtime type > GPU, then rerun this cell.')

## Install compatible libraries

Colab supplies CUDA-enabled PyTorch; keep that build and install the pinned Diffusers stack.

In [ ]:
%pip install -q diffusers==0.30.3 transformers==4.44.2 accelerate==0.34.2 'huggingface-hub>=0.23.2,<1.0' 'numpy<2.0' 'Pillow>=10.0,<12.0' imageio==2.35.1 imageio-ffmpeg==0.5.1

## Load the pretrained model once

The model download is large and can take several minutes. CPU offload and attention slicing reduce GPU memory use.

In [ ]:
import torch
from diffusers import DiffusionPipeline, DPMSolverMultistepScheduler

MODEL_ID = 'cerspense/zeroscope_v2_576w'
if torch.cuda.is_available() and torch.cuda.get_device_properties(0).total_memory < 8 * 1e9:
    raise RuntimeError('This model needs about 8 GB VRAM; select a larger Colab GPU.')
dtype = torch.float16 if torch.cuda.is_available() else torch.float32
pipe = DiffusionPipeline.from_pretrained(MODEL_ID, torch_dtype=dtype, use_safetensors=False)
pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
if torch.cuda.is_available():
    pipe.enable_model_cpu_offload()
    pipe.enable_attention_slicing('auto')
else:
    pipe.to('cpu')
print('Model ready:', MODEL_ID)

## Enter a prompt and settings

Use a scene + subject + action + camera + lighting/style. Defaults are deliberately short for a first run. More frames and steps cost more memory/time.

In [ ]:
prompt = 'A small boat sailing across a calm lake at sunrise, gentle water movement, cinematic wide shot.'
num_frames = 24       # supported in this practical: 8 to 32; model trained at 24
fps = 8                # playback rate; duration = frames / FPS
num_inference_steps = 25  # practical range: 10 to 50
guidance_scale = 9.0   # practical range: 1 to 15
seed = 42              # use -1 for a new random seed

assert isinstance(prompt, str) and prompt.strip(), 'Prompt cannot be empty.'
assert len(prompt.strip()) <= 1000, 'Prompt must be at most 1000 characters.'
assert isinstance(num_frames, int) and 8 <= num_frames <= 32
assert isinstance(fps, int) and 1 <= fps <= 24
assert isinstance(num_inference_steps, int) and 10 <= num_inference_steps <= 50
assert isinstance(guidance_scale, (int, float)) and 1 <= guidance_scale <= 15
assert isinstance(seed, int) and -1 <= seed <= 4294967295
print('Duration:', round(num_frames / fps, 2), 'seconds')

## Generate frames and save an actual MP4

Generation is computationally demanding and may take several minutes. This cell checks the returned frames and verifies the encoded MP4 is present and non-empty.

In [ ]:
from pathlib import Path
from datetime import datetime
import re, imageio.v2 as imageio

resolved_seed = seed if seed >= 0 else int(torch.randint(0, 4294967296, (1,)).item())
generator = torch.Generator(device='cuda' if torch.cuda.is_available() else 'cpu').manual_seed(resolved_seed)
try:
    with torch.inference_mode():
        result = pipe(prompt.strip(), height=320, width=576, num_frames=num_frames,
                      num_inference_steps=num_inference_steps, guidance_scale=float(guidance_scale),
                      generator=generator)
except RuntimeError as exc:
    if 'out of memory' in str(exc).lower():
        if torch.cuda.is_available(): torch.cuda.empty_cache()
        raise RuntimeError('GPU out of memory; reduce frames/steps or select a larger GPU.') from exc
    raise

frames = result.frames[0]
assert len(frames) == num_frames, f'Expected {num_frames} frames, received {len(frames)}.'
assert frames[0].shape[:2] == (320, 576), f'Unexpected frame size: {frames[0].shape[:2]}'
slug = '_'.join(re.findall(r'[a-zA-Z0-9]+', prompt.lower())[:5]) or 'video'
output_dir = Path('/content/outputs')
output_dir.mkdir(parents=True, exist_ok=True)
video_path = output_dir / f'{slug}_{datetime.now():%Y%m%d_%H%M%S}.mp4'
try:
    imageio.mimsave(video_path, frames, fps=fps, codec='libx264', format='FFMPEG',
                    pixelformat='yuv420p', macro_block_size=1)
except (OSError, RuntimeError, ValueError) as exc:
    raise RuntimeError('MP4 encoding failed; verify imageio-ffmpeg is installed.') from exc
assert video_path.is_file() and video_path.stat().st_size > 0, 'MP4 was not saved successfully.'
reader = imageio.get_reader(video_path, format='FFMPEG')
try:
    decoded_frame = reader.get_data(0)
    assert reader.count_frames() == num_frames, 'Encoded video frame count does not match.'
    assert decoded_frame.shape == (320, 576, 3), f'Unexpected decoded size: {decoded_frame.shape}'
finally:
    reader.close()
print('Frames:', len(frames), '| Frame size: 576x320 | FPS:', fps)
print('Saved:', video_path, '| Bytes:', video_path.stat().st_size, '| Seed:', resolved_seed)

## Preview the generated video

In [ ]:
from IPython.display import Video, display
display(Video(str(video_path), embed=True, width=576))

## Download the MP4

Only run after the encoding cell reports a non-empty output file.

In [ ]:
from google.colab import files
files.download(str(video_path))

## Optional: copy the result to Google Drive

In [ ]:
# Uncomment to save a copy in Drive:
# from google.colab import drive
# import shutil
# drive.mount('/content/drive')
# drive_path = Path('/content/drive/MyDrive/generated_video.mp4')
# shutil.copy2(video_path, drive_path)
# print('Copied to:', drive_path)

## Try different prompts

Change `prompt` and rerun the settings, generation, preview, and download cells. Keep the same seed to compare prompt changes; set `seed = -1` for a new random seed.

1. A butterfly flying through a colorful garden, flowers moving gently in the breeze.
2. A futuristic city at night with flying vehicles, glowing neon lights, and slow camera movement.
3. Waves gently hitting a beach at sunset, warm lighting and a peaceful atmosphere.
4. A small robot walking through a green forest, cinematic lighting and smooth motion.
